In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report, roc_auc_score
from imblearn.over_sampling import SMOTE
from collections import Counter
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import lightgbm as lgb

In [2]:
# df_original = pd.read_csv('Children Recode_final.csv')
# df_original['Malnurished'] = df_original[['Underweight', 'Stunting', 'Wasting']].max(axis=1)
# df = df_original.drop(['Underweight', 'Stunting', 'Wasting'], axis = 1)
# df.head()

# X = df.drop(columns=['Malnurished'])
# y = df['Malnurished']

# # Train-test Split
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state= 12)

# # Columns to scale
# columns_to_scale = ['Child_age', 'Age_first_sex', 'BMI', 'Mother_age_current', 'Mother_age_at_first_birth']
# scaler = StandardScaler()

# # Make copies of training and test sets
# X_train_scaled = X_train.copy()
# X_test_scaled = X_test.copy()

# # Scale only selected columns
# X_train_scaled[columns_to_scale] = scaler.fit_transform(X_train[columns_to_scale])
# X_test_scaled[columns_to_scale] = scaler.transform(X_test[columns_to_scale])

# # Apply SMOTE
# sm = SMOTE(random_state=42)
# X_train_sm, y_train_sm = sm.fit_resample(X_train_scaled, y_train)

# print("Before SMOTE:", Counter(y_train))
# print("After SMOTE:", Counter(y_train_sm))

# # Compute class imbalance ratio
# from collections import Counter
# counter = Counter(y_train)
# scale_pos_weight = counter[0] / counter[1]

# xgb = XGBClassifier(
#     scale_pos_weight=scale_pos_weight,
#     n_estimators=100,
#     max_depth=5,
#     learning_rate=0.1,
#     eval_metric='logloss'
# )

# xgb.fit(X_train_sm, y_train_sm)

# # Predict on test set
# y_pred = xgb.predict(X_test_scaled)

# # Evaluate
# print("Classification Report on Test Set:")
# print(classification_report(y_test, y_pred))

# # Train LightGBM
# lgbm = lgb.LGBMClassifier(
#     n_estimators=500,
#     learning_rate=0.05,
#     max_depth=6,
#     class_weight='balanced',  # LightGBM auto-balancing
#     random_state=42
# )

# lgbm.fit(X_train_sm, y_train_sm)

# # Predict and Evaluate
# y_pred = lgbm.predict(X_test_scaled)

# print("Classification Report on Test Set:")
# print(classification_report(y_test, y_pred))


In [3]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, recall_score, roc_auc_score
from xgboost import XGBClassifier

In [4]:
# Load Data
df_original = pd.read_csv('df.csv')
df_original['Malnurished'] = df_original[['Underweight', 'Stunting', 'Wasting']].max(axis=1)
df = df_original.drop(['Underweight', 'Stunting', 'Wasting'], axis = 1)

# Train-test Split
X = df.drop(columns=['Malnurished'])
y = df['Malnurished']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, stratify = y, random_state = 42)

In [5]:
# Step 5: Calculate the scale_pos_weight parameter
# Count negative (0) and positive (1) samples in y_train:
neg, pos = np.bincount(y_train)
scale_pos_weight = neg / pos
print("Scale_pos_weight:", scale_pos_weight)

# Step 6: Initialize the XGBClassifier with the scale_pos_weight parameter
xgb = XGBClassifier(eval_metric='logloss', scale_pos_weight=scale_pos_weight, random_state=42)

# Define the parameter grid for hyperparameter tuning
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.1, 0.01],
    'min_child_weight': [1, 3],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# Setup GridSearchCV, focusing on a recall-based scoring metric
grid_search = GridSearchCV(estimator=xgb,
                           param_grid=param_grid,
                           scoring='recall',
                           cv=5,
                           n_jobs=-1,
                           verbose=2)

# Fit GridSearchCV on the training data
grid_search.fit(X_train, y_train)

# Output the best parameters and best CV recall score
print("Best Parameters:", grid_search.best_params_)
print("Best CV Recall Score:", grid_search.best_score_)

Scale_pos_weight: 1.9597315436241611
Fitting 5 folds for each of 96 candidates, totalling 480 fits
Best Parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 3, 'min_child_weight': 1, 'n_estimators': 100, 'subsample': 0.8}
Best CV Recall Score: 0.5571428571428572


**XGBoost Model**

In [9]:
best_xgb = XGBClassifier(colsample_bytree= 0.8, learning_rate= 0.1, max_depth= 3, min_child_weight= 3, 
                        n_estimators= 100, subsample= 1.0)

# Fit GridSearchCV on the training data
best_xgb.fit(X_train, y_train)

# Evaluate the best estimator on the test set
# best_xgb = grid_search.best_estimator_
y_pred = best_xgb.predict(X_test)
y_pred_prob = best_xgb.predict_proba(X_test)[:, 1]

# Calculate metrics
test_recall = recall_score(y_test, y_pred)
test_roc_auc = roc_auc_score(y_test, y_pred_prob)

print("Test ROC AUC:", test_roc_auc)
print(f'Confusion Matrix: \n{pd.crosstab(y_test, y_pred)}\n')
print("Classification Report:\n", classification_report(y_test, y_pred))

Test ROC AUC: 0.6283786889767399
Confusion Matrix: 
col_0          0   1
Malnurished         
0            272  20
1            129  20

Classification Report:
               precision    recall  f1-score   support

           0       0.68      0.93      0.78       292
           1       0.50      0.13      0.21       149

    accuracy                           0.66       441
   macro avg       0.59      0.53      0.50       441
weighted avg       0.62      0.66      0.59       441



**CatBoost Model**

**Feature Importance Plot**

In [7]:
pd.Series(lgbm.feature_importances_, index=X_train.columns).sort_values().plot(kind='barh', figsize=(10, 6))
plt.title("LightGBM Feature Importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()


NameError: name 'lgbm' is not defined